# `results_from_multiagents.csv` — `result` 컬럼 JSON 파싱·정제

`result`는 JSON 배열 문자열입니다. pandas가 CSV를 읽으면 따옴표 이스케이프는 자동으로 풀립니다.

In [ ]:
import json
import pandas as pd
from pathlib import Path

RESULTS_CSV = Path("/Users/baebichna/git_repo/multiagents/results_from_multiagents.csv")
df = pd.read_csv(RESULTS_CSV)
df.head()

In [ ]:
def parse_triples_cell(raw) -> list[dict]:
    """result 셀 → list[dict] (비어있거나 NaN이면 [])"""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    if isinstance(raw, list):
        return raw
    s = str(raw).strip()
    if not s or s.lower() == "nan":
        return []
    return json.loads(s)


df["triples"] = df["result"].apply(parse_triples_cell)
df["triple_count"] = df["triples"].apply(len)
df[["idx", "item", "triple_count", "error"]].head(10)

In [ ]:
# 한 행에서 head/relation/tail만 담은 깔끔한 DataFrame (explode)
exploded = df.explode("triples", ignore_index=True)
triples_df = pd.json_normalize(exploded["triples"])
flat = pd.concat(
    [
        exploded[["idx", "item", "sentence", "error"]].reset_index(drop=True),
        triples_df[["head", "relation", "tail"]],
    ],
    axis=1,
)
flat = flat.dropna(subset=["head"])  # triples가 빈 행 제거
flat.head(15)

In [ ]:
# (선택) 긴 문자열 하나로 — 예: 한 아이템당 탭 구분 triple 줄
def triples_to_lines(triples: list[dict]) -> str:
    lines = []
    for t in triples:
        lines.append(f"{t['head']}\t{t['relation']}\t{t['tail']}")
    return "\n".join(lines)


df["triples_tsv"] = df["triples"].apply(triples_to_lines)
df.loc[0, "triples_tsv"][:500]

In [ ]:
# (선택) 정제본 저장 — triple이 풀어진 long 형
OUT_FLAT = RESULTS_CSV.with_name("results_from_multiagents_triples_flat.csv")
flat.to_csv(OUT_FLAT, index=False, encoding="utf-8")
print("saved:", OUT_FLAT)